# SupplyGuard: Google Colab Training Pipeline
## End-to-End Multi-Seed Training Harness on A100 GPU (or T4 GPU)

Welcome to the **SupplyGuard** training notebook. SupplyGuard is a real-time multi-echelon supply chain risk alert system featuring hybrid Spatio-Temporal Graph Convolutional Networks (ST-GCN + LSTM) and paper-style baselines.

### Overview of Training Workflow:
1. **GPU Diagnostics**: Verify GPU availability (e.g., NVIDIA T4 Tensor Core GPU) and CUDA drivers.
2. **Repository Acquisition**: Clone the authoritative GitHub repository (`https://github.com/Sanath1121/Supply-Guard.git`).
3. **Environment Setup**: Install pinned dependencies from `requirements.txt`.
4. **Data Acquisition & Verification**: Download the raw SCRM time series dataset (Mendeley Data V2) and verify SHA-256 integrity, timestamp monotonicity, and gap profiles.
5. **5-Seed Grid Training**: Train all 4 model architectures across 5 random seeds (42–46) on GPU with early stopping, gradient clipping, and checkpoint-resume resilience.
6. **Artifact Archiving**: Compress model checkpoints (`outputs/models/`) and training loss/summary metrics (`outputs/results/`) into `outputs.zip`.
7. **Artifact Download**: Automatically trigger browser download of `outputs.zip` to retrieve trained model weights and logs.

> **Colab Runtime Note**: Please ensure hardware accelerator is set to **GPU** (`Runtime -> Change runtime type -> Hardware accelerator -> A100 GPU (or T4 GPU)`).

### Step 1: Hardware & CUDA Diagnostics
Verify that the Colab runtime is allocated an NVIDIA GPU (recommended: Tesla T4, V100, or A100) and that CUDA drivers are operational.

In [ ]:
!nvidia-smi

### Step 1b: Connect Google Drive (Optional & Recommended for Overnight Runs)
Mount your Google Drive now before training starts. Authorizing now allows the notebook to automatically back up `outputs.zip` to your Google Drive when training completes, even if your laptop is closed, disconnected, or shut down.

In [ ]:
# Mount Google Drive early so you can authorize it before you step away
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive mounted successfully!")

### Step 2: Clone Authoritative SupplyGuard Repository
Clone the official SupplyGuard GitHub repository and navigate into the project directory.

In [ ]:
!git clone https://github.com/Sanath1121/Supply-Guard.git
%cd Supply-Guard

### Step 3: Install Required Dependencies
Install project dependencies pinned in `requirements.txt`. Colab pre-installs compatible PyTorch and CUDA packages; this step installs essential libraries including `statsmodels`, `networkx`, and scikit-learn.

In [ ]:
!pip install -r requirements.txt

### Step 4: Dataset Acquisition & Cryptographic Verification
Download the official SCRM dataset (`SCRM_timeSeries_2018_train.csv`) from the pinned GitHub commit and perform Phase 0 verification:
- Cryptographic SHA-256 validation (`d2e71ae7f55fa70ef498fecb9b6db0c9fd59688f17f8ad3c27c7576f09e76ff3`).
- Chronological timestamp parsing and gap analysis.
- Null profile check (~5.1% in Distributor, ~5.5% in Total Cost).

In [ ]:
!python setup_and_download.py

### Step 5: Execute 5-Seed Multi-Architecture Training Grid
Train all 4 model variants across all 5 seeds (`42, 43, 44, 45, 46`) on the GPU:
- `lstm`: Graph-free LSTM baseline (temporal ablation)
- `paper_overall`: Re-implementation of the IEEE ICCMC 2025 hybrid paper baseline
- `st_gcn_lstm_sym`: STGCN-LSTM with symmetric Kipf-Welling adjacency
- `st_gcn_lstm_dir`: STGCN-LSTM with directed upstream/downstream adjacency matrices

**Resilience Features**:
- **Automatic Checkpoint Resumption**: If a Colab session disconnects, re-running this cell automatically skips already trained checkpoints (`outputs/models/{model}_seed{seed}.pt`).
- **Gradient Norm Clipping**: Bounded at `cfg.GRAD_CLIP = 1.0`.
- **Validation Early Stopping**: Monitored on validation loss with `patience = 10`.
- **Progressive Logging**: Per-epoch loss history logged to `outputs/results/{model}_seed{seed}_loss.csv` and summary with wall-clock time saved to `outputs/results/training_summary.csv`.

In [ ]:
!python -m training.train --device cuda --batch-size 256

### Step 6: Package Trained Artifacts into Archive
Compress trained PyTorch model checkpoints and training evaluation metrics into a single zip archive (`outputs.zip`).

In [ ]:
!zip -r outputs.zip outputs/models outputs/results

### Step 7: Download Artifacts to Local Environment
Automatically initiate browser download of `outputs.zip` to transfer trained weights and logs to your local workstation for Phase 5 evaluation and Streamlit dashboard replay.

*Note: If the download does not trigger automatically, verify that browser pop-ups are allowed for colab.research.google.com, or manually download `outputs.zip` from the Files panel in the left sidebar.*

In [ ]:
import os
import shutil
from google.colab import drive, files

# --- 1. Save permanently to Google Drive ---
drive_dest = "/content/drive/MyDrive/outputs.zip"
if not os.path.exists("/content/drive/MyDrive"):
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive mount notice:", e)

if os.path.exists("/content/drive/MyDrive"):
    try:
        shutil.copy("outputs.zip", drive_dest)
        print(f"SUCCESS: outputs.zip saved to your Google Drive: {drive_dest}")
    except Exception as e:
        print("Error copying to Google Drive:", e)
else:
    print("Google Drive not mounted; skipping Drive backup.")

# --- 2. Trigger browser download to local PC ---
try:
    files.download("outputs.zip")
    print("SUCCESS: Browser download triggered for outputs.zip.")
except Exception as e:
    print("Browser download notice (normal if laptop was asleep/closed):", e)
